# VA Analysis LangGraph V03

This notebook contains a clean LangGraph prototype for:
- Node 01: extract issues and target path hints from Jenkins / Trivy output
- Node 02: fetch dependency files, resolve likely target paths, and update only `bom_friendly_fix`

It uses the existing `dev_test/py/non_ai_flow/github_service.py` helper for repository file fetches.


In [ ]:
import os
import re
import sys
from pathlib import Path
from pprint import pprint
from typing import Annotated, Literal, Sequence, TypedDict

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langgraph.graph.state import Command, END, START, StateGraph

load_dotenv()
llm_model = os.getenv("LLM_MODEL")
llm = ChatOpenAI(model=llm_model)


In [ ]:
class IssueCore(TypedDict):
    issue_category: Literal["VA", "OTHER"]
    detailed_issue: str
    initial_fix: str


class DetailedIssue(TypedDict):
    issue_category: Literal["VA", "OTHER"]
    detailed_issue: str
    initial_fix: str
    bom_friendly_fix: str


class TargetPathHint(TypedDict):
    issue_index: int
    target_file_hint: str


class TargetPathInfo(TypedDict):
    issue_index: int
    target_file_hint: str
    resolved_path: str
    confidence: str


class IssueExtractionResult(TypedDict):
    issue_list: list[IssueCore]
    target_path_hints: list[TargetPathHint]


class BomFixOnly(TypedDict):
    bom_friendly_fix: str


class Node02Result(TypedDict):
    issue_list: list[BomFixOnly]


class VaAnalysisState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    issue_list: list[DetailedIssue]
    target_path_hints: list[TargetPathHint]
    target_paths: list[TargetPathInfo]


In [ ]:
print(Path.cwd())
jenkins_output_raw_text = Path("trivy_scan_input.txt").read_text(encoding="utf-8")
jenkins_output_raw_text[:2500]


In [ ]:
def _collect_message_text(messages: Sequence[BaseMessage]) -> str:
    parts: list[str] = []
    for msg in messages:
        content = getattr(msg, "content", "")
        if isinstance(content, str):
            parts.append(content)
        elif isinstance(content, list):
            for item in content:
                if isinstance(item, str):
                    parts.append(item)
                elif isinstance(item, dict):
                    text = item.get("text")
                    if isinstance(text, str):
                        parts.append(text)
    return "\n".join(parts)


def _extract_file_hints(text: str) -> list[str]:
    patterns = [
        r"(?i)\b[\w./-]*pom\.xml\b",
        r"(?i)\b[\w./-]*build\.gradle(?:\.kts)?\b",
        r"(?i)\b[\w./-]*gradle\.properties\b",
        r"(?i)\b[\w./-]*settings\.gradle(?:\.kts)?\b",
        r"(?i)\b[\w./-]*dockerfile\b",
    ]
    hints: set[str] = set()
    for pattern in patterns:
        for match in re.findall(pattern, text or ""):
            hints.add(match.replace("\\", "/").strip())
    return sorted(hints)


def _get_fetch_analysis_files():
    try:
        from non_ai_flow.github_service import fetch_analysis_files
        return fetch_analysis_files
    except ImportError:
        project_root = Path.cwd().resolve().parents[1]
        py_root = project_root / "dev_test" / "py"
        if str(py_root) not in sys.path:
            sys.path.append(str(py_root))
        from non_ai_flow.github_service import fetch_analysis_files
        return fetch_analysis_files


def _filter_relevant_files(files: dict[str, str]) -> list[dict[str, str]]:
    relevant_files: list[dict[str, str]] = []
    for path, content in files.items():
        lower = path.lower()
        if (
            lower.endswith("pom.xml")
            or lower.endswith("build.gradle")
            or lower.endswith("build.gradle.kts")
            or lower.endswith("dockerfile")
            or lower.endswith("gradle.properties")
            or lower.endswith("settings.gradle")
            or lower.endswith("settings.gradle.kts")
        ):
            relevant_files.append({"path": path, "content": content})
    return relevant_files


def _score_repo_file(path: str, content: str, hints: list[str], jenkins_text: str) -> int:
    lower_path = path.lower()
    content_lower = content.lower()
    score = 0

    for hint in hints:
        hint_lower = hint.lower()
        if lower_path == hint_lower:
            score += 100
        elif lower_path.endswith(hint_lower):
            score += 60
        elif hint_lower in lower_path:
            score += 25

    jenkins_lower = jenkins_text.lower()
    if "pom.xml" in jenkins_lower and lower_path.endswith("pom.xml"):
        score += 15
    if "build.gradle" in jenkins_lower and "build.gradle" in lower_path:
        score += 15
    if "dockerfile" in jenkins_lower and lower_path.endswith("dockerfile"):
        score += 15

    depth = lower_path.count("/")
    score += max(0, 10 - depth)

    if lower_path.endswith("pom.xml"):
        if "<dependencymanagement>" in content_lower:
            score += 20
        if "<parent>" in content_lower:
            score += 15

    if lower_path.endswith(("build.gradle", "build.gradle.kts")):
        if "platform(" in content_lower or "enforcedplatform(" in content_lower:
            score += 20
        if "dependencymanagement" in content_lower:
            score += 10

    if lower_path.endswith("dockerfile") and content_lower.startswith("from"):
        score += 10

    return score


def _resolve_target_paths(
    issue_count: int,
    target_path_hints: list[TargetPathHint],
    relevant_files: list[dict[str, str]],
    jenkins_text: str,
) -> list[TargetPathInfo]:
    if not relevant_files:
        return []

    resolved: list[TargetPathInfo] = []
    grouped_hints: dict[int, list[str]] = {}
    for hint in target_path_hints:
        grouped_hints.setdefault(hint["issue_index"], []).append(hint["target_file_hint"])

    generic_hints = _extract_file_hints(jenkins_text)

    for issue_index in range(issue_count):
        issue_hints = grouped_hints.get(issue_index, []) or generic_hints or [relevant_files[0]["path"]]
        ranked_files = sorted(
            relevant_files,
            key=lambda item: _score_repo_file(item["path"], item["content"], issue_hints, jenkins_text),
            reverse=True,
        )
        best = ranked_files[0]
        best_score = _score_repo_file(best["path"], best["content"], issue_hints, jenkins_text)
        confidence = "high" if best_score >= 80 else "medium" if best_score >= 35 else "low"

        resolved.append({
            "issue_index": issue_index,
            "target_file_hint": issue_hints[0],
            "resolved_path": best["path"],
            "confidence": confidence,
        })

    return resolved


In [ ]:
# NODE 01 -> fetch and analyze all issues
# Updates only issue_category, detailed_issue, initial_fix
# Also stores path hints for Node 02
def issue_extraction(state: VaAnalysisState):
    issue_extraction_agent_system_prompt = """
    You are Node 01 in a LangGraph vulnerability remediation workflow.

    Responsibilities:
    1. Read the Jenkins / Trivy scan output.
    2. Extract all issues.
    3. Flag each issue as VA or OTHER.
    4. Produce a detailed issue description.
    5. Produce an initial fix.
    6. Provide a target_file_hint for each issue when possible.

    Rules:
    - Do not produce bom_friendly_fix here.
    - Use zero-based issue_index values in target_path_hints.
    - If many issues come from the same target, repeat the same hint for each issue.
    """

    issue_extraction_agent = create_agent(
        model=llm,
        response_format=IssueExtractionResult,
        system_prompt=issue_extraction_agent_system_prompt,
    )

    issue_extraction_agent_response = issue_extraction_agent.invoke({
        "messages": state["messages"]
    })

    pprint(issue_extraction_agent_response)

    core_issues = issue_extraction_agent_response["structured_response"]["issue_list"]
    hint_items = issue_extraction_agent_response["structured_response"].get("target_path_hints", [])

    full_issues: list[DetailedIssue] = [
        {
            "issue_category": issue["issue_category"],
            "detailed_issue": issue["detailed_issue"],
            "initial_fix": issue["initial_fix"],
            "bom_friendly_fix": "",
        }
        for issue in core_issues
    ]

    if not hint_items:
        jenkins_text = _collect_message_text(state["messages"])
        fallback_hints = _extract_file_hints(jenkins_text)
        if fallback_hints:
            hint_items = [
                {
                    "issue_index": idx,
                    "target_file_hint": fallback_hints[0],
                }
                for idx in range(len(full_issues))
            ]

    return Command(update={
        "issue_list": full_issues,
        "target_path_hints": hint_items,
    })


In [ ]:
# NODE 02 -> fetch dependency/container files and update only bom_friendly_fix
# Uses dev_test/py/non_ai_flow/github_service.py
def dependency_file_analysis(state: VaAnalysisState, config=None):
    configurable = (config or {}).get("configurable", {})
    repo_name = configurable.get("repo_name")
    branch_name = configurable.get("branch_name", "main")
    mock_files = configurable.get("mock_files")
    existing_issues = state.get("issue_list", [])
    target_path_hints = state.get("target_path_hints", [])
    jenkins_text = _collect_message_text(state.get("messages", []))

    if mock_files:
        files = mock_files
    else:
        if not repo_name:
            return Command(update={
                "issue_list": [
                    {
                        **issue,
                        "bom_friendly_fix": issue.get("bom_friendly_fix", "") or (
                            "Node 02 skipped because repo_name was not provided in config."
                        ),
                    }
                    for issue in existing_issues
                ],
                "target_paths": [],
            })

        fetch_analysis_files = _get_fetch_analysis_files()
        files = fetch_analysis_files(repo_name, branch_name)

    relevant_files = _filter_relevant_files(files)
    resolved_target_paths = _resolve_target_paths(
        issue_count=len(existing_issues),
        target_path_hints=target_path_hints,
        relevant_files=relevant_files,
        jenkins_text=jenkins_text,
    )

    selected_paths = []
    for item in resolved_target_paths:
        if item["resolved_path"] not in selected_paths:
            selected_paths.append(item["resolved_path"])

    selected_files = [
        item for item in relevant_files if item["path"] in selected_paths
    ]
    if not selected_files:
        selected_files = relevant_files[:5]

    issue_lines = []
    for idx, issue in enumerate(existing_issues, start=1):
        issue_lines.append(
            f"Issue {idx}\n"
            f"- issue_category: {issue.get('issue_category', '')}\n"
            f"- detailed_issue: {issue.get('detailed_issue', '')}\n"
            f"- initial_fix: {issue.get('initial_fix', '')}\n"
            f"- current_bom_friendly_fix: {issue.get('bom_friendly_fix', '')}"
        )

    target_path_lines = []
    for item in resolved_target_paths:
        target_path_lines.append(
            f"- issue_index={item['issue_index']}, hint={item['target_file_hint']}, "
            f"resolved_path={item['resolved_path']}, confidence={item['confidence']}"
        )

    file_lines = []
    for item in selected_files:
        preview = item["content"][:8000]
        file_lines.append(f"### FILE: {item['path']}\n{preview}")

    node02_prompt = f"""
You are Node 02 in a LangGraph vulnerability remediation workflow.

Task:
Review the extracted issues and the repository dependency/container files.
For each issue, improve ONLY the `bom_friendly_fix` field.

Rules:
1. Do not change issue_category.
2. Do not change detailed_issue.
3. Do not change initial_fix.
4. Update ONLY bom_friendly_fix.
5. Make the fix BOM-friendly and non-conflicting.
6. Prefer a single source of truth:
   - Spring Boot parent/BOM
   - Maven dependencyManagement
   - Gradle platform/version catalog/property
   - Docker base image strategy if relevant
7. Avoid scattered direct overrides when a parent/BOM/platform controls the version.
8. Keep output aligned issue-for-issue with the input order.
9. Use the resolved target paths below as the most likely file locations for this Jenkins response.

Resolved target paths:
{chr(10).join(target_path_lines) if target_path_lines else '- No path resolutions available.'}

Issues:
{chr(10).join(issue_lines) if issue_lines else '- No issues provided.'}

Most relevant dependency and container files:
{chr(10).join(file_lines) if file_lines else '- No dependency files found.'}

Return only a structured issue_list where each item contains:
- bom_friendly_fix
"""

    node02_agent = create_agent(
        model=llm,
        response_format=Node02Result,
        system_prompt=(
            "You improve only the bom_friendly_fix field for each issue, using "
            "dependency files to produce BOM-friendly, non-conflicting guidance."
        ),
    )

    node02_response = node02_agent.invoke({
        "messages": [HumanMessage(content=node02_prompt)]
    })

    pprint({
        "resolved_target_paths": resolved_target_paths,
        "selected_paths": [item["path"] for item in selected_files],
        "node02_response": node02_response,
    })

    updated_fix_items = node02_response["structured_response"]["issue_list"]

    merged_issue_list = []
    for idx, issue in enumerate(existing_issues):
        new_fix = issue.get("bom_friendly_fix", "")
        if idx < len(updated_fix_items):
            new_fix = updated_fix_items[idx].get("bom_friendly_fix", new_fix)

        merged_issue_list.append({
            **issue,
            "bom_friendly_fix": new_fix,
        })

    return Command(update={
        "issue_list": merged_issue_list,
        "target_paths": resolved_target_paths,
    })


In [ ]:
# Graph setup
graph = StateGraph(VaAnalysisState)

graph.add_node(issue_extraction, "issue_extraction")
graph.add_node(dependency_file_analysis, "dependency_file_analysis")

graph.add_edge(START, "issue_extraction")
graph.add_edge("issue_extraction", "dependency_file_analysis")
graph.add_edge("dependency_file_analysis", END)

compiled_graph = graph.compile()
compiled_graph


## Ways To Test Properly

### Test 1: Offline mock test
- Use `mock_files` so you can validate Node 02 without GitHub access.
- Confirm Node 01 extracts issues and path hints.
- Confirm Node 02 updates only `bom_friendly_fix`.
- Confirm `target_paths` is populated with resolved paths.

### Test 2: Ambiguous multi-module test
- Provide more than one `pom.xml` or `build.gradle` in `mock_files`.
- Confirm `target_paths` chooses the most relevant path using the Jenkins hint and scoring.

### Test 3: Real GitHub repo test
- Set `repo_name` and `branch_name` in graph config.
- Make sure `PAYLOAD_MONITOR_GITHUB_TOKEN` is available.
- Confirm `fetch_analysis_files()` loads real repo manifests.

### Test 4: Missing config test
- Run Node 02 without `repo_name`.
- Confirm it does not crash and instead leaves issues intact with a skip message.


In [ ]:
# Test 1: Offline mock test
mock_files = {
    "pom.xml": """
<project>
  <parent>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-parent</artifactId>
    <version>2.7.18</version>
  </parent>
  <dependencies>
    <dependency>
      <groupId>org.springframework.boot</groupId>
      <artifactId>spring-boot-starter-web</artifactId>
    </dependency>
  </dependencies>
</project>
""",
    "service-a/pom.xml": """
<project>
  <dependencies>
    <dependency>
      <groupId>org.apache.logging.log4j</groupId>
      <artifactId>log4j-core</artifactId>
      <version>2.14.1</version>
    </dependency>
  </dependencies>
</project>
""",
    "Dockerfile": "FROM eclipse-temurin:17-jre\nRUN echo hello\n",
}

mock_results = compiled_graph.invoke(
    {
        "messages": [HumanMessage(content=jenkins_output_raw_text)],
        "issue_list": [],
        "target_path_hints": [],
        "target_paths": [],
    },
    config={
        "configurable": {
            "mock_files": mock_files,
        }
    },
)

pprint(mock_results["target_paths"])
pprint(mock_results["issue_list"][:3])


In [ ]:
# Test 2: Real GitHub repo test
# Replace owner/repo and branch before running.

real_results = compiled_graph.invoke(
    {
        "messages": [HumanMessage(content=jenkins_output_raw_text)],
        "issue_list": [],
        "target_path_hints": [],
        "target_paths": [],
    },
    config={
        "configurable": {
            "repo_name": "owner/repo",
            "branch_name": "main",
        }
    },
)

pprint(real_results["target_paths"])
pprint(real_results["issue_list"][:3])
